In [ ]:
from pathlib import Path

import pandas as pd

candidates = [
    Path("data/processed/wem_5min_panel.parquet"),
    Path("../data/processed/wem_5min_panel.parquet"),
]
panel_path = next(p for p in candidates if p.exists())

df = pd.read_parquet(panel_path, engine="pyarrow")

df["interval_end"] = pd.to_datetime(df["interval_end"])
df["trading_interval_end"] = pd.to_datetime(df["trading_interval_end"])
df = df.sort_values("interval_end").reset_index(drop=True)

print(f"Loaded: {panel_path}")
print(f"Shape: {df.shape}")
print(df.head())


Loaded: ../data/processed/wem_5min_panel.parquet
Shape: (302964, 33)
         interval_end     mcp  cr_raise  cr_lower  reg_raise  reg_lower  \
0 2023-10-01 08:00:00  738.00   1180.27      0.49     577.15        0.5   
1 2023-10-01 08:05:00  738.00   1107.77      0.49     577.15        0.5   
2 2023-10-01 08:10:00  738.00    689.17      0.10     575.44        0.5   
3 2023-10-01 08:15:00  738.00    684.83      0.10     575.44        0.5   
4 2023-10-01 08:20:00  263.54    190.69      0.10     104.88        0.5   

   rocof  operational_demand_mw  unscheduled_demand_mw  \
0    0.0                1606.73                1598.12   
1    0.0                1594.24                1586.53   
2    0.0                1581.83                1574.04   
3    0.0                1554.51                1547.28   
4    0.0                1524.89                1517.86   

   operational_withdrawal_mw  ...  residual_demand_mw mcp_lag1  mcp_lag6  \
0                      -8.61  ...             1191.34  

In [3]:
DROP_COLS = [
    "rocof",                    # nearly empty
    "minute",                   # no useful signal for MCP
    "dpv_mw_30min",             # redundant with dpv_mw
    "sent_out_mwh",             # duplicate of sent_out_mw
    "sent_out_mw",              # redundant with demand / scada
    "scada_mwh",                # duplicate of scada_mw
    "unscheduled_demand_mw",    # near-duplicate of operational_demand_mw
    "scheduled_energy_mw",      # near-duplicate of scada_mw
    "trading_interval_end",     # join key only
    "is_weekend",               # deterministic from dow
]

missing_before = (df.isna().mean() * 100).sort_values(ascending=False)
print("Missing values before dropping:")
print(missing_before[missing_before > 0].round(3).to_string())

missing_drop = [c for c in DROP_COLS if c not in df.columns]
if missing_drop:
    raise KeyError(f"Columns not found in data: {missing_drop}")

model_df = df.drop(columns=DROP_COLS).copy()
model_df = model_df.set_index("interval_end").sort_index()

print(f"Remaining columns: {list(model_df.columns)}")
print(f"Final shape: {model_df.shape}")

missing_after = (model_df.isna().mean() * 100).sort_values(ascending=False)
print("Missing values after dropping:")
print(missing_after[missing_after > 0].round(4).to_string() or "(none)")

Missing values before dropping:
dpv_mw_30min          5.135
sent_out_mwh          2.853
sent_out_mw           2.853
mcp_lag12             0.004
mcp_lag6              0.002
rtp                   0.002
residual_demand_mw    0.000
dpv_mw                0.000
mcp_lag1              0.000
Remaining columns: ['mcp', 'cr_raise', 'cr_lower', 'reg_raise', 'reg_lower', 'operational_demand_mw', 'operational_withdrawal_mw', 'dpv_mw', 'rtp', 'stem_price', 'stem_qty_mwh', 'stem_bid_mwh', 'stem_offer_mwh', 'scada_mw', 'residual_demand_mw', 'mcp_lag1', 'mcp_lag6', 'mcp_lag12', 'hour', 'dow', 'month', 'year']
Final shape: (302964, 22)
Missing values after dropping:
mcp_lag12             0.0040
mcp_lag6              0.0020
rtp                   0.0017
dpv_mw                0.0003
mcp_lag1              0.0003
residual_demand_mw    0.0003


In [4]:
# Keep the useful model frame
TARGET = "mcp"
FEATURES = [
    c for c in model_df.columns
    if c not in {TARGET, "rtp", "cr_raise", "cr_lower", "reg_raise", "reg_lower"}
]

# Create lagged versions of contemporaneous price variables so they are valid for 5-minute-ahead forecasting.
model_df["rtp_lag1"] = model_df["rtp"].shift(1)
for col in ["cr_raise", "cr_lower", "reg_raise", "reg_lower"]:
    model_df[f"{col}_lag1"] = model_df[col].shift(1)

FEATURES = [
    c for c in model_df.columns
    if c not in {TARGET, "rtp", "cr_raise", "cr_lower", "reg_raise", "reg_lower"}
]

work = model_df[[TARGET, *FEATURES]].copy()
print(f"Model table: {work.shape}")
work.head()
## Final cleaned data

Model table: (302964, 22)


,mcp,operational_demand_mw,operational_withdrawal_mw,dpv_mw,stem_price,stem_qty_mwh,stem_bid_mwh,stem_offer_mwh,scada_mw,residual_demand_mw,...,mcp_lag12,hour,dow,month,year,rtp_lag1,cr_raise_lag1,cr_lower_lag1,reg_raise_lag1,reg_lower_lag1
interval_end,,,,,,,,,,,,,,,,,,,,,
2023-10-01 08:00:00,738.00,1606.73,-8.61,415.39,65.25,84.415,675.462,2181.218,1609.092,1191.34,...,NaN,8,6,10,2023,NaN,NaN,NaN,NaN,NaN
2023-10-01 08:05:00,738.00,1594.24,-7.71,427.91,65.19,80.206,623.392,2208.662,1586.340,1166.33,...,NaN,8,6,10,2023,555.99,1180.27,0.49,577.15,0.5
2023-10-01 08:10:00,738.00,1581.83,-7.79,460.49,65.19,80.206,623.392,2208.662,1573.584,1121.34,...,NaN,8,6,10,2023,102.04,1107.77,0.49,577.15,0.5
2023-10-01 08:15:00,738.00,1554.51,-7.23,486.06,65.19,80.206,623.392,2208.662,1564.812,1068.45,...,NaN,8,6,10,2023,102.04,689.17,0.10,575.44,0.5
2023-10-01 08:20:00,263.54,1524.89,-7.03,493.67,65.19,80.206,623.392,2208.662,1542.444,1031.22,...,NaN,8,6,10,2023,102.04,684.83,0.10,575.44,0.5


In [5]:
work = work.dropna()
print(f"Shape after removing NaN: {work.shape}")

Shape after removing NaN: (302947, 22)


In [ ]:
from statsmodels.tsa.statespace.sarimax import SARIMAX

# Use the target series only, indexed by timestamp
mcp_ts = work["mcp"].sort_index().asfreq("5min")

# Fit a seasonal SARIMA model on the 5-minute MCP series
# Daily seasonality: 5 minutes * 24 hours * 12 = 1440? No, 5 minutes * 24 hours = 288.
sarima_model = SARIMAX(
    mcp_ts,
    order=(1, 0, 1),
    seasonal_order=(1, 0, 1, 288),
    enforce_stationarity=False,
    enforce_invertibility=False,
)

sarima_fit = sarima_model.fit(disp=False)

print(sarima_fit.summary())